<a href="https://colab.research.google.com/github/salih-salu/Speech_Emotion_Recognition_using_AudioFiles/blob/main/Test_speech_emotion_recognition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import librosa
from tensorflow.keras.models import load_model

In [2]:
def add_noise(audio, noise_factor=0.005):
    noise = np.random.randn(len(audio))
    return audio + noise_factor * noise

def shift_audio(audio, shift_max=0.2):
    shift = int(np.random.uniform(-shift_max, shift_max) * len(audio))
    return np.roll(audio, shift)

def change_pitch(audio, sr, n_steps=2):
    return librosa.effects.pitch_shift(
        y=audio,
        sr=sr,
        n_steps=n_steps
    )


def extract_mfcc(file_path, n_mfcc=40, max_pad_len=174):

    audio, sample_rate = librosa.load(
        file_path,
        sr=16000
    )

    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sample_rate,
        n_mfcc=n_mfcc
    )

    # Padding
    if mfcc.shape[1] < max_pad_len:

        pad_width = max_pad_len - mfcc.shape[1]

        mfcc = np.pad(
            mfcc,
            ((0, 0), (0, pad_width)),
            mode='constant'
        )

    # Trimming
    else:
        mfcc = mfcc[:, :max_pad_len]

    return mfcc

def extract_mfcc_augmented(file_path, n_mfcc=40, max_pad_len=174):

    audio, sr = librosa.load(
        file_path,
        sr=16000
    )

    # Random augmentation
    choice = np.random.choice(
        ["original", "noise", "shift", "pitch"]
    )

    if choice == "noise":
        audio = add_noise(audio)

    elif choice == "shift":
        audio = shift_audio(audio)

    elif choice == "pitch":
        audio = change_pitch(audio, sr, n_steps=2)

    # Extract MFCC
    mfcc = librosa.feature.mfcc(
        y=audio,
        sr=sr,
        n_mfcc=n_mfcc
    )

    # Padding / Trimming
    if mfcc.shape[1] < max_pad_len:

        pad_width = max_pad_len - mfcc.shape[1]

        mfcc = np.pad(
            mfcc,
            ((0, 0), (0, pad_width)),
            mode='constant'
        )

    else:
        mfcc = mfcc[:, :max_pad_len]

    return mfcc

In [3]:
emotion_names = [
    'neutral',
    'calm',
    'happy',
    'sad',
    'angry',
    'fearful',
    'disgust',
    'surprised'
]

In [4]:
model = load_model('/content/best_model_54 (1).keras')

In [5]:
audio_path = '/content/03-01-08-02-02-01-21.wav'
mfcc = extract_mfcc(audio_path)
mfcc = mfcc[np.newaxis, ..., np.newaxis]

prediction = model.predict(mfcc)
predicted_class = np.argmax(prediction)
print("Predicted class:", predicted_class)
print("Predicted emotion:", emotion_names[predicted_class])

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 432ms/step
Predicted class: 7
Predicted emotion: surprised
